# M3 — TimeGrad on Colab (the conditional-diffusion centerpiece)

M0/M1/M2 were a repeated value, a linear-Gaussian ARIMA, and an RNN with a parametric head. **M3 TimeGrad** (Rasul et al., 2021) is the model the whole ladder builds toward: at every step it runs a **conditional denoising diffusion model** — the exact generative family the PML course derives in its final chapter — made *conditional on the past* via an autoregressive RNN.

Like the M2 notebook this is a thin driver — all logic lives in the repo (`src/models/timegrad.py`, `experiments/run_timegrad.py`). It clones the repo, installs the **pinned** heavy group, trains TimeGrad on the **Exchange** train split, scores the **test** windows with the *same* metrics module as M0–M2, and appends one row to `results/registry.csv`.

> **⚠️ Read this first.** TimeGrad runs on **PyTorchTS**, whose version coupling to GluonTS is famously fragile. PyPI `pytorchts` is dead (it imports a removed GluonTS path), and PyTorchTS **`master`** has since moved on to require GluonTS ≥0.14; so we install from a **pinned PyTorchTS commit** (`81be06bcc`, the last one compatible with GluonTS 0.13) against a **pinned** `gluonts==0.13.7` + `numpy==1.26.4`. Downgrading numpy **requires a one-time runtime restart** (section 2). Follow the sections in order; the restart step tells you exactly what to do.

**Before you start:** `Runtime → Change runtime type → GPU` (TimeGrad sampling is the expensive rung — a GPU matters here).

## 1. Clone the sandbox repo

In [ ]:
!git clone -q https://github.com/Icaica14/pml-diffusion-tsf-test.git
%cd pml-diffusion-tsf-test

## 2. Install the pinned heavy group (the fragile part)

**Recipe A** — the only combo with evidence of working on a fresh Colab today:

- Pin `numpy==1.26.4`, `pandas==2.1.4`, `gluonts==0.13.7` (PyTorchTS needs gluonts in the 0.11–0.13 range — **not** ≥0.14), plus `holidays<0.40` and `pydantic<2` (old GluonTS breaks on the v2 lines).
- Add **`pytorch_lightning`** explicitly: `gluonts==0.13.7`'s `gluonts.torch` backend imports it at module load (TimeGrad's estimator pulls it in), and our `--no-deps` PyTorchTS install won't fetch it for us.
- Install **PyTorchTS at a pinned commit** (`81be06bcc`) with `--no-deps`. We do **not** use `master`: commit #168 (2024-06-14) bumped PyTorchTS's imports to the gluonts **0.14+** layout (`gluonts.torch.distributions.output`), which doesn't exist in 0.13.7 — so `master` now fails the import smoke-test. `81be06bcc` is the parent of that commit: the last state that still imports `PtArgProj` from `gluonts.torch.distributions.distribution_output`. `--no-deps` also stops its stale `numpy~=1.16` / `pandas~=1.1` pins from re-breaking the env.
- Keep Colab's preinstalled **torch** (it works; let pts use it).

This downgrades numpy, so **the runtime must restart once** afterwards (next cell explains).

In [ ]:
!pip install -q "numpy==1.26.4" "pandas==2.1.4" "gluonts==0.13.7" "pytorch_lightning" "holidays<0.40" "pydantic<2"
# PyTorchTS pinned to the LAST commit before #168 (2024-06-14) bumped its gluonts
# imports to the 0.14+ layout (`gluonts.torch.distributions.output`). That commit
# breaks against our pinned gluonts==0.13.7; this parent commit is the last state
# that imports PtArgProj from `gluonts.torch.distributions.distribution_output`.
!pip install -q --no-deps git+https://github.com/zalandoresearch/pytorch-ts.git@81be06bcc128729ad8901fcf1c722834f176ac34

## ⚠️ RESTART THE RUNTIME NOW

The cell above downgraded numpy while numpy 2.x was already loaded in memory. You **must** restart for the new version to take effect:

1. **`Runtime → Restart session`** (or `Runtime → Restart runtime`).
2. After it restarts, **continue from section 3 below** — run those cells.
3. **Do _not_ re-run cells 1, 2 or the install cell.** A restart keeps your installed packages and the cloned files on disk; only the Python kernel resets. Re-running them just wastes time.

(If you skip the restart you'll hit `numpy.dtype size changed, Expected 96 ... got 88` — that means numpy 2.x is still live in memory.)

## 3. After the restart — re-enter the repo + sanity check

The restart reset the working directory back to `/content`, so we `cd` into the repo again and confirm the pinned versions + GPU.

In [ ]:
%cd /content/pml-diffusion-tsf-test
import numpy as np, pandas as pd, gluonts, torch
import pts
print('numpy   ', np.__version__, '(must be 1.26.x)')
print('pandas  ', pd.__version__)
print('gluonts ', gluonts.__version__, '(must be 0.13.x)')
print('torch   ', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
from pts.model.time_grad import TimeGradEstimator  # import smoke-test
print('TimeGradEstimator import: OK')

## 4. Train + evaluate M3

One global TimeGrad over the 8 channels modelled **jointly** (the multivariate bridge `ds.to_gluonts_multivariate`); the test windows are then forecast with no refit (each conditioned only on its own context — the same leakage-free protocol as M1/M2).

Two things worth knowing:
- **`input_size` is auto-detected.** TimeGrad's RNN input width is a notorious must-match magic number; the wrapper starts from a guess and self-corrects from the shape check, so you normally don't touch it. If you ever see a shape error naming an expected size, re-run adding `--input-size <that number>`.
- **This is the expensive rung.** Diffusion *sampling* (S trajectories × τ steps × `diff_steps` denoising steps × 1430 windows) is the cost the project's thesis is about. The default below is sized to finish in a reasonable time on a GPU; if predict is slow, drop `--samples` or `--diff-steps` for a first pass.

In [ ]:
!python -m experiments.run_timegrad --epochs 30 --samples 100 --diff-steps 100 --device cuda

## 5. Inspect the results and bring the row home

The cloned repo's `results/registry.csv` already holds the committed M0, M1 and M2 rows; the cell above appended **`timegrad`**. Read it back to see the whole ladder side by side, then download the file and send it over to record + commit (handled from the laptop, no AI co-author trailer).

In [ ]:
import pandas as pd
df = pd.read_csv('results/registry.csv')
cols = ['model', 'MASE', 'CRPS', 'cov50', 'cov90', 'width90', 'fit_s', 'predict_s']
print(df[[c for c in cols if c in df.columns]].to_string(index=False))

In [ ]:
# Download the updated registry so you can commit it from your laptop.
from google.colab import files
files.download('results/registry.csv')